In [11]:
import pandas as pd, numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
import pingouin as pg
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

csv_path = '/home/mw/input/data_total74037403/国家角度数据集/国家.csv'
years = range(2010, 2023)

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

all_feats = ['移动网络覆盖率','固定宽带普及率','ICT服务出口','信息通讯服务出口','科技期刊文章',
             '高等教育劳动力占比','ICT相关论文发表量','政府效率','独角兽公司数量',
             '研发支出占GDP比重','教育支出总额占比','研发人员（每百万人）','高等教育入学率']

gai_feats = ['固定宽带普及率','移动网络覆盖率','研发支出占GDP比重','研发人员（每百万人）',
             '独角兽公司数量','科技期刊文章','ICT相关论文发表量',
             '高等教育劳动力占比','高等教育入学率']

df = pd.read_csv(csv_path, encoding='utf-8')
for c in all_feats: df[c] = pd.to_numeric(df[c], errors='coerce')
df = df.sort_values(['国家名称','year'])
df[all_feats] = df.groupby('国家名称')[all_feats]\
                  .transform(lambda g: g.interpolate('linear', limit_direction='both'))
df[all_feats] = df[all_feats].fillna(df[all_feats].mean())

def gini(arr):
    arr = np.array(arr).flatten()
    if np.amin(arr) < 0:
        arr -= np.amin(arr)
    arr += 1e-8  # 防止为0
    arr = np.sort(arr)
    n = arr.size
    cumarr = np.cumsum(arr)
    return (n + 1 - 2 * np.sum(cumarr) / cumarr[-1]) / n

def critic_weight(mat):
    std = mat.std(0, ddof=1)
    corr = np.corrcoef(mat, rowvar=False)
    info = std * (1 - np.abs(corr)).sum(1)
    return info / info.sum()

tei = []
for y in years:
    sub = df[df.year==y]
    X = MinMaxScaler().fit_transform(sub[all_feats])
    w = critic_weight(X)
    score = (X*w).sum(1)
    tei.append(1 - gini(score)) # TEI的均衡度 (1 - Gini系数)
tei = np.array(tei)

gai_pca1 = []
for y in years:
    sub = df[df.year==y]
    X1 = StandardScaler().fit_transform(sub[gai_feats])
    pc1 = PCA(n_components=1).fit_transform(X1).ravel()
    pc1_pos = MinMaxScaler().fit_transform(pc1.reshape(-1,1)).ravel()
    gai_pca1.append(pc1_pos.mean()) # GAI的年度平均水平

gai_pca1  = np.array(gai_pca1)

idx = pd.DataFrame({'year':years,'TEI':tei,
                    'GAI_PCA1':gai_pca1})

print("\n年度指数：")
print(idx.round(4))

print("\n\n========== 回归分析结果 ==========")

for col in ['GAI_PCA1']:
    
    print(f"\n因变量: {col}  | 自变量: TEI (科技平权指数的均衡度)")
    print("----------------------------------------------------")

    # --- 方法1: 普通最小二乘法 (OLS) ---
    res_ols_pg = pg.linear_regression(idx['TEI'], idx[col])
    print(f"\n[方法1: 普通最小二乘法 (OLS) - 使用 pingouin]")
    print(res_ols_pg.round(4))

    # --- 方法2: 多项式回归 (例如，二次多项式) ---
    # 使用 statsmodels.formula.api 进行，可以方便地指定多项式项
    # 公式: GAI_PCA1 ~ TEI + TEI^2
    formula_poly = f"{col} ~ TEI + I(TEI**2)"
    poly_model_sm = smf.ols(formula=formula_poly, data=idx).fit()
    print(f"\n[方法2: 多项式回归 (二次) - 使用 statsmodels]")
    print(f"模型公式: {formula_poly}")
    print("系数表:")
    print(poly_model_sm.summary().tables[1])
    print(f"R-squared: {poly_model_sm.rsquared:.4f}")
    print(f"Adj. R-squared: {poly_model_sm.rsquared_adj:.4f}")

    # --- 方法3: 稳健线性回归 (Robust Linear Models - RLM) ---
    # 使用 statsmodels.api 进行，对异常值不那么敏感
    # M=sm.robust.norms.HuberT() 使用Huber's T M-estimation
    X_rlm = sm.add_constant(idx['TEI'])  # 需要手动添加截距项
    y_rlm = idx[col]
    
    rlm_model = sm.RLM(y_rlm, X_rlm, M=sm.robust.norms.HuberT()).fit()
    print(f"\n[方法3: 稳健线性回归 (RLM, Huber's T) - 使用 statsmodels]")
    print("系数表:")
    print(rlm_model.summary().tables[1])



年度指数：
    year     TEI  GAI_PCA1
0   2010  0.7997    0.4228
1   2011  0.8021    0.4411
2   2012  0.8074    0.4379
3   2013  0.8133    0.4456
4   2014  0.8196    0.4723
5   2015  0.8246    0.4672
6   2016  0.8254    0.4547
7   2017  0.8246    0.4810
8   2018  0.8283    0.5105
9   2019  0.8364    0.5587
10  2020  0.8420    0.6198
11  2021  0.8397    0.6679
12  2022  0.8440    0.6791


========== 回归分析结果 ==========

因变量: GAI_PCA1  | 自变量: TEI (科技平权指数的均衡度)
----------------------------------------------------

[方法1: 普通最小二乘法 (OLS) - 使用 pingouin]
       names    coef      se       T    pval      r2  adj_r2  CI[2.5%]  \
0  Intercept -3.8911  0.7189 -5.4124  0.0002  0.7733  0.7527   -5.4735   
1        TEI  5.3463  0.8728  6.1257  0.0001  0.7733  0.7527    3.4253   

   CI[97.5%]  
0    -2.3088  
1     7.2672  

[方法2: 多项式回归 (二次) - 使用 statsmodels]
模型公式: GAI_PCA1 ~ TEI + I(TEI**2)
系数表:
                  coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------

c:\Users\Lenovo\anaconda3\envs\weiAI\lib\site-packages\scipy\stats\_axis_nan_policy.py:531: UserWarning:

kurtosistest only valid for n>=20 ... continuing anyway, n=13



In [14]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
import plotly.express as px

csv_path = '/home/mw/input/data_total74037403/国家角度数据集/国家.csv'
years     = range(2010, 2023)

all_feats = ['移动网络覆盖率','固定宽带普及率','ICT服务出口','信息通讯服务出口','科技期刊文章',
             '高等教育劳动力占比','ICT相关论文发表量','政府效率','独角兽公司数量',
             '研发支出占GDP比重','教育支出总额占比','研发人员（每百万人）','高等教育入学率']

gai_feats = ['固定宽带普及率','移动网络覆盖率','研发支出占GDP比重','研发人员（每百万人）',
             '独角兽公司数量','科技期刊文章','ICT相关论文发表量',
             '高等教育劳动力占比','高等教育入学率']

bubble_size_col = '研发支出占GDP比重'

# ------------------------- 读数与预处理 -------------------------
df = pd.read_csv(csv_path, encoding='utf-8')

# 数值列转为 float
for col in set(all_feats + gai_feats):
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 升序排序
df = df.sort_values(['国家名称', 'year'])

df[all_feats] = df.groupby('国家名称')[all_feats]\
                  .transform(lambda g: g.interpolate('linear', limit_direction='both'))
df[all_feats] = df[all_feats].fillna(df[all_feats].mean())

def critic_weight(mat: np.ndarray) -> np.ndarray:
    """
    计算 CRITIC 权重
    mat: 已归一化后的二维矩阵 (n_sample, n_feat)
    """
    std  = mat.std(axis=0, ddof=1)
    corr = np.corrcoef(mat, rowvar=False)
    info = std * (1 - np.abs(corr)).sum(axis=1)
    w    = info / info.sum()
    return w

tei_list = []
gai_list = []

for y in years:
    sub_idx = df['year'] == y
    sub_df  = df.loc[sub_idx, :]

    # ---------- TEI ----------
    X_all = MinMaxScaler().fit_transform(sub_df[all_feats])     # 按列 0-1 归一
    w_all = critic_weight(X_all)                                # CRITIC 权重
    tei_score = (X_all * w_all).sum(axis=1)                     # 每行得分
    tei_list.extend(tei_score)

    # ---------- GAI ----------
    X_gai  = StandardScaler().fit_transform(sub_df[gai_feats])  # Z-score
    pc1    = PCA(n_components=1).fit_transform(X_gai).ravel()
    gai_score = MinMaxScaler().fit_transform(pc1.reshape(-1, 1)).ravel()
    gai_list.extend(gai_score)

# 将结果写回原表
df['TEI']       = tei_list
df['GAI_PCA1']  = gai_list

fig = px.scatter(
        data_frame   = df,
        x            = 'TEI',
        y            = 'GAI_PCA1',
        animation_frame = 'year',
        animation_group = '国家名称',
        size         = bubble_size_col,
        color        = '国家名称',
        hover_name   = '国家名称',
        hover_data   = {
            'year'        : True,
            'TEI'         : ':.3f',
            'GAI_PCA1'    : ':.3f',
            '研发支出占GDP比重': False,
        },
        size_max     = 40,
        height       = 500,
        labels       = {
            'TEI': '科技平权 (TEI)',
            'GAI_PCA1': '生成式 AI (GAI)'
        }
)

fig.update_layout(
        title = '各国 TEI vs. GAI (交互式气泡图)',
        xaxis = dict(range=[0, 1]),
        yaxis = dict(range=[0, 1]),
        legend_title_text = '国家名称',
        font = dict(family='Microsoft YaHei', size=12)
)

fig.show()